# 02 — Modèle de capteur

y = T_air + v. Ici v = bruit **et** quantification.
Le salon marche par **0,1 °C** : un 21,83 °C vrai devient 21,8 °C lu.
Ce n'est **pas** une constante de temps des murs.

Code : `basic_mpc.data.sensors`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from basic_mpc.data.sensors import SensorModel, infer_resolution, quantize_measurement
from basic_mpc.models.plant import ThermalPlant, literature_plant_params, synthetic_weather

capteur = SensorModel(name="livingroom", resolution=0.1, unit="celsius")
print(capteur.observation_equation())

vrai = np.linspace(19.02, 21.47, 80)
lu = np.array([quantize_measurement(v, 0.1) for v in vrai])
print("résolution inférée sur y =", infer_resolution(lu))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(vrai, label="T_air vraie", color="#3d6b6b")
ax.step(np.arange(len(lu)), lu, where="mid", label="y (0,1 °C)", color="#8c4a32")
ax.set_xlabel("index")
ax.set_ylabel("°C")
ax.set_title("La marche d'escalier est le capteur, pas le RC")
ax.legend(frameon=False)
plt.show()

Sur le **plant**, `observe()` ajoute un bruit puis quantifie. Comparer état vrai et y (figure S1).

In [ ]:
params = literature_plant_params()
n = int(24 * 3600 / params.dt_seconds)
w = synthetic_weather(n, params.dt_seconds, seed=2)
plant = ThermalPlant(params=params, x0=np.array([20.0, 20.0]), seed=0)
traj = plant.simulate(w["t_ext"].to_numpy(), w["S"].to_numpy(), w["P"].to_numpy())

fig, ax = plt.subplots(figsize=(8, 3.2))
t_h = np.arange(n) * params.dt_seconds / 3600
ax.plot(t_h, traj["ta_true"], color="#3d6b6b", label="T_air vraie")
ax.plot(t_h, traj["y"], color="#8c4a32", lw=0.8, label="y quantifiée")
ax.set_xlabel("heures")
ax.set_ylabel("°C")
ax.legend(frameon=False)
plt.show()